<div style="padding: 20px; background-color: #f0f7ff; border-radius: 10px; border-left: 8px solid #2196F3;">
    <h1 style="color: #0d47a1; margin-bottom: 5px;">🚀 Building AI Agents with LangChain</h1>
    <p style="color: #1565c0; font-size: 1.1em;">Mastering Autonomous Reasoning & Tool Integration</p>
</div>

> *A comprehensive guide for technical interviews and production reference.*

### 🧠 1. What is an AI Agent?

An **AI Agent** is an autonomous system that takes a high-level goal and independently navigates the path to achievement.

| Feature | Description |
| :--- | :--- |
| 🎯 **Goal-Driven** | You provide the *What*, it handles the *How*. |
| 🧩 **Planning** | Breaks complex requests into logical sub-tasks. |
| 🛠️ **Tool Usage** | Interacts with APIs, DBs, and Search Engines. |
| 💾 **Memory** | Maintains context across multi-step executions. |
| 🔄 **Adaptive** | Re-evaluates strategy if a tool or step fails. |

---

### 🏗️ 2. The Anatomy of an Agent

An agent lives at the intersection of **Intelligence** and **Action**:

1.  **Reasoning Engine (LLM):** The "Brain". Responsible for intent analysis and strategy.
2.  **Tools:** The "Hands". Interfaces (APIs, Functions) that allow the LLM to impact the real world.

---

### 🔁 3. The ReAct Pattern

**Re**asoning + **Act**ing = **ReAct**. This is the standard loop for autonomous logic.

> **The Cycle:**
> 1. **Thought:** "I need to find X to solve Y."
> 2. **Action:** "I will use the Search Tool."
> 3. **Observation:** "The search result says Z."

✨ *Benefit: The internal scratchpad makes the agent's logic fully auditable and transparent.*

---

### 💻 4. Implementation in LangChain

To manifest an agent in code, you require these three pillars:

*   **Tools:** `@tool` decorated functions or standard classes.
*   **Agent Object:** The blueprint created via `create_react_agent`.
*   **Agent Executor:** The **runtime** that manages the loop and the memory scratchpad.

---

### 🌐 5. The Future: From Linear to Graph

⚠️ **Note:** `AgentExecutor` is now considered "Legacy" for complex production apps.

*   **The Problem:** Linear execution is brittle and hard to control.
*   **The Solution: LangGraph.** Industry is moving toward stateful, cyclic graphs that allow for fine-grained control and multi-agent collaboration.

---

In [5]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
import requests
from langchain_community.tools import DuckDuckGoSearchRun

C:\Users\SAUNISH69\AppData\Local\Temp\ipykernel_21464\994026605.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


In [6]:
search_tool = DuckDuckGoSearchRun()

In [7]:
@tool
def get_weather_data(city: str) -> str:
  """
  This function fetches the current weather data for a given city
  """
  url = f'https://api.weatherstack.com/current?access_key=4d1d8ae207a8c845a52df8a67bf3623e&query={city}'

  response = requests.get(url)

  return response.json()

In [8]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.0)

In [12]:
from langchain_classic.agents import create_react_agent, AgentExecutor
from langchain_classic import hub

In [13]:
# Step 2: Pull the ReAct prompt from LangChain Hub
prompt = hub.pull("hwchase17/react")  # pulls the standard ReAct agent prompt

#we are using pre defined ReAct prompt from langchain hub but we can also define our own, To use predefined is recomended as its optimized.

C:\Users\SAUNISH69\AppData\Local\Temp\ipykernel_21464\585618604.py:2: LangChainDeprecationWarning: langchain_classic.hub.pull is deprecated. Use the LangSmith SDK instead.
  prompt = hub.pull("hwchase17/react")  # pulls the standard ReAct agent prompt


In [14]:
# Step 3: Create the ReAct agent manually with the pulled prompt
agent = create_react_agent(
    llm=llm,
    tools=[search_tool, get_weather_data],
    prompt=prompt
)

In [15]:
# Step 4: Wrap it with AgentExecutor
agent_executor = AgentExecutor(
    agent=agent,
    tools=[search_tool, get_weather_data],
    verbose=True
)

In [16]:
# Step 5: Invoke
response = agent_executor.invoke({"input": "Find the capital of Madhya Pradesh, then find it's current weather condition"})
print(response)



> Entering new AgentExecutor chain...
Action: duckduckgo_search
Action Input: capital of Madhya Pradesh2 days ago - Its capital is Bhopal. Other major cities include Indore, Gwalior, Jabalpur, Chhindwara, and Sagar. Madhya Pradesh is the second largest Indian state by area and the fifth largest state by population with over 72 million residents. It borders the states of Rajasthan to the northwest, Uttar ... 2 days ago - Bhopal (Hindi: Bhōpāl, pronounced [bʱoːpaːl] ⓘ) is the capital city of the Indian state of Madhya Pradesh and the administrative headquarters of both Bhopal district and Bhopal division. 1 week ago - Jabalpur (IPA: [d͡ʒəbəlpʊɾ]), formerly anglicised as Jubbulpore, is a city situated on the banks of the Narmada River in the state of Madhya Pradesh, India. Jabalpur is the administrative headquarters of the Jabalpur district and the Jabalpur ... 6 days ago - It has no coastline and no international frontier. Its physiography is characterized by low hills, extensive plate

In [17]:
response['output']

'The capital of Madhya Pradesh is Bhopal. I am unable to retrieve its current weather condition due to a usage limit on the weather data tool.'